# Hito 1

Executable notebook for the Capstone (F1 Race Strategy Advisor).

In [87]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator
from sklearn.metrics import brier_score_loss, roc_auc_score
RANDOM_STATE = 414

In [88]:
# Load data
df = pd.read_csv('f1_strategy_race_level.csv')

print(f'Shape: {df.shape}')
print('Columns:')
df.columns.tolist()

display(df.head(3))
for col in df.columns:
    print(f'{col}: {df[col].dtype}')

Shape: (2447, 47)
Columns:


,season,round,race_name,circuit_id,circuit,circuit_type,driver_id,driver_name,Driver,Team,...,avg_track_temp,avg_air_temp,finish_position,points,positions_gained,is_top3,is_top5,is_top10,dnf,status
0,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,bottas,Bottas,BOT,Mercedes,...,40.300000,23.329091,1,26.0,1.0,1,1,1,0,Finished
1,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,hamilton,Hamilton,HAM,Mercedes,...,40.260000,23.330909,2,18.0,-1.0,1,1,1,0,Finished
2,2019,1,Australian Grand Prix,albert_park,Australian Grand Prix,semi-street,max_verstappen,Verstappen,VER,Red Bull,...,40.276364,23.334545,3,15.0,1.0,1,1,1,0,Finished


season: int64
round: int64
race_name: object
circuit_id: object
circuit: object
circuit_type: object
driver_id: object
driver_name: object
Driver: object
Team: object
constructor_name: object
grid_position: float64
qualifying_position: float64
qualifying_time_s: float64
driver_prior3_avg_finish: float64
constructor_prior3_avg_finish: float64
driver_circuit_prior_avg: float64
constructor_tier: object
n_stops: int64
strategy_type: object
compound_sequence: object
stint_lengths: object
stint1_length: int64
stint2_length: int64
stint3_length: int64
stint4_length: int64
stint5_length: int64
avg_pit_stop_duration_s: float64
total_pit_time_s: float64
first_pit_lap: float64
last_pit_lap: float64
track_status_summary: object
safety_car_periods: int64
safety_car_laps: int64
vsc_laps: int64
weather_actual: object
wet_laps: int64
avg_track_temp: float64
avg_air_temp: float64
finish_position: int64
points: float64
positions_gained: float64
is_top3: int64
is_top5: int64
is_top10: int64
dnf: int64
st

In [89]:
# Minimal validations and baseline feature definition
df['season'] = pd.to_numeric(df['season'], errors='coerce')
df['is_top10'] = pd.to_numeric(df['is_top10'], errors='coerce').fillna(0).astype(int)
df['grid_position'] = pd.to_numeric(df['grid_position'], errors='coerce')


feature_cols = ['grid_position', 'constructor_tier']
target_col = 'is_top10'

print('Baseline features used (pre-race observable only):')
print(feature_cols)

Baseline features used (pre-race observable only):
['grid_position', 'constructor_tier']


#### Weather Context: Abandoned Approach

**Decision: We abandoned the weather context feature approach.**

**Rationale:**
- The dataset lacks a dedicated weather feature that can serve as a reliable pre-race proxy.
- The `weather_actual` column represents post-race conditions, not Friday-observable forecasts.
- Analysis showed that including a derived weather proxy (e.g., `wet_day_forecast_proxy` from `weather_actual`) does not meaningfully improve model performance.
- Model metrics remain stable whether or not the weather variable is included, indicating minimal predictive value from weather encoding in this dataset.
- To maintain a defensible baseline, we stick to universally pre-race observable features: grid position and constructor tier.

**Conclusion:**
The baseline model uses only grid position and constructor tier. Strategy features (pit stops, tire compound, stint lengths) remain scenario inputs for what-if analysis, not baseline predictors.

In [90]:
# Locked temporal split for Hito 1
train_mask = df['season'].isin([2019, 2020, 2021])
calib_mask = df['season'] == 2022
test_mask = df['season'].isin([2023, 2024])

train_df = df.loc[train_mask].copy()
calib_df = df.loc[calib_mask].copy()
test_df = df.loc[test_mask].copy()

print(f"Train (2019-2021): {train_df.shape[0]} rows")
print(f"Calibration (2022): {calib_df.shape[0]} rows")
print(f"Test (2023-2024): {test_df.shape[0]} rows")

Train (2019-2021): 1132 rows
Calibration (2022): 426 rows
Test (2023-2024): 889 rows


In [91]:
# Baseline model pipeline definition
numeric_features = ['grid_position']
categorical_features = ['constructor_tier']

numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

base_model = Pipeline(steps=[
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))
])

X_train, y_train = train_df[feature_cols], train_df[target_col]
X_calib, y_calib = calib_df[feature_cols], calib_df[target_col]
X_test, y_test = test_df[feature_cols], test_df[target_col]

base_model.fit(X_train, y_train)

# Calibration on 2022 block (without touching test), current sklearn API
frozen_base = FrozenEstimator(base_model)
calibrated_model = CalibratedClassifierCV(estimator=frozen_base, method='isotonic')
calibrated_model.fit(X_calib, y_calib)

print('Baseline model trained on 2019-2021 and validated on 2022.')

Baseline model trained on 2019-2021 and validated on 2022.


In [92]:
# Final evaluation on test 2023-2024 (single look)
proba_test = calibrated_model.predict_proba(X_test)[:, 1]
proba_test = np.clip(proba_test, 1e-6, 1 - 1e-6)

brier = brier_score_loss(y_test, proba_test)
auc = roc_auc_score(y_test, proba_test)

print('Metrics on test (2023-2024):')
print(f'- Brier score {brier:.4f}')
print(f'- ROC-AUC:    {auc:.4f}')

print('\nDocent reference: Brier=0.137, ROC-AUC=0.892')
print('Grid-rule baseline: Brier=0.208')

Metrics on test (2023-2024):
- Brier score 0.1450
- ROC-AUC:    0.8694

Docent reference: Brier=0.137, ROC-AUC=0.892
Grid-rule baseline: Brier=0.208


The value that we get was 0.145, that is still a bit behind the docent value but is a notable upgrade from the grid rule 0.208, with a 34% of improve.

In [93]:
# Leakage audit: separate pre-race, scenario inputs, and audit/post-race columns
pre_race_features = [
    'grid_position',
    'constructor_tier',
    'driver_id'
]

scenario_input_columns = [
    'n_stops',
    'compound_sequence',
    'stint_lengths',
    'stint1_length',
    'stint2_length',
    'stint3_length',
    'stint4_length',
    'stint5_length',
    'avg_pit_stop_duration_s'
]

audit_or_postrace_columns = [
    'finish_position',
    'points',
    'positions_gained',
    'safety_car_periods',
    'safety_car_laps',
    'vsc_laps',
    'weather_actual',
    'wet_laps',
    'status',
    'dnf'
]

target_column = 'is_top10'

leakage_audit = {
    'target': target_column,
    'used_in_baseline_model': pre_race_features,
    'note_on_weather_context': 'Weather context (weather_actual) was evaluated as a potential feature but abandoned. No meaningful improvement in model performance. Dataset lacks reliable pre-race weather forecast data.',
    'scenario_inputs_not_used_for_baseline_fit': [
        c for c in scenario_input_columns if c in df.columns
    ],
    'audit_or_postrace_columns_not_used_for_baseline_fit': [
        c for c in audit_or_postrace_columns if c in df.columns
    ],
    'note': 'Strategy features treated as scenario inputs. Baseline uses only universally pre-race observable features to maintain defensibility.'
}

pd.Series(leakage_audit, dtype='object')

target                                                                                          is_top10
used_in_baseline_model                                      [grid_position, constructor_tier, driver_id]
note_on_weather_context                                Weather context (weather_actual) was evaluated...
scenario_inputs_not_used_for_baseline_fit              [n_stops, compound_sequence, stint_lengths, st...
audit_or_postrace_columns_not_used_for_baseline_fit    [finish_position, points, positions_gained, sa...
note                                                   Strategy features treated as scenario inputs. ...
dtype: object

In [94]:
# Scenario evaluation from framing.md
# The modeled baseline uses only pre-race observable variables: grid_position and constructor_tier.

scenario_rows = pd.DataFrame([
    {
        'scenario': 'A_controlled_baseline',
        'grid_position': 12,
        'constructor_tier': 'midfield',
        'driver_id': 'HAM',
        'n_stops': 1,
        'compound_sequence': 'S-M',
        'stint_lengths': '40',
        'avg_pit_stop_duration_s': 22
    },
    {
        'scenario': 'B_strategy_change',
        'grid_position': 12,
        'constructor_tier': 'midfield',
        'driver_id': 'HAM',
        'n_stops': 2,
        'compound_sequence': 'S-M-S',
        'stint_lengths': '22-20',
        'avg_pit_stop_duration_s': 22
    },
    {
        'scenario': 'C_context_alternative',
        'grid_position': 12,
        'constructor_tier': 'midfield',
        'driver_id': 'HAM',
        'n_stops': 2,
        'compound_sequence': 'S-M-S',
        'stint_lengths': '20-18',
        'avg_pit_stop_duration_s': 26
    }
])

scenario_rows['p_top10_calibrated'] = calibrated_model.predict_proba(scenario_rows[feature_cols])[:, 1]

# Primary delta requested in framing: A vs B
p_a = scenario_rows.loc[scenario_rows['scenario'] == 'A_controlled_baseline', 'p_top10_calibrated'].iloc[0]
p_b = scenario_rows.loc[scenario_rows['scenario'] == 'B_strategy_change', 'p_top10_calibrated'].iloc[0]
delta_a_minus_b = p_a - p_b

display_cols = [
    'scenario',
    'grid_position',
    'constructor_tier',
    'driver_id',
    'n_stops',
    'compound_sequence',
    'stint_lengths',
    'avg_pit_stop_duration_s',
    'p_top10_calibrated'
]

print(f'Baseline features: grid_position, constructor_tier, driver_id (pre-race observable only).')
print(f'Delta P(top10) [A - B]: {delta_a_minus_b:+.4f}')
scenario_rows[display_cols]

Baseline features: grid_position, constructor_tier, driver_id (pre-race observable only).
Delta P(top10) [A - B]: +0.0000


,scenario,grid_position,constructor_tier,driver_id,n_stops,compound_sequence,stint_lengths,avg_pit_stop_duration_s,p_top10_calibrated
0,A_controlled_baseline,12,midfield,HAM,1,S-M,40,22,0.604938
1,B_strategy_change,12,midfield,HAM,2,S-M-S,22-20,22,0.604938
2,C_context_alternative,12,midfield,HAM,2,S-M-S,20-18,26,0.604938


## Notes for report alignment

- This notebook respects the locked temporal split and uses 2022 only for calibration.
- The baseline avoids direct leakage from post-race variables.
- Strategy variables are documented as scenario inputs (what-if).
- For Hito 2, compare against a second model (e.g., Random Forest).